# DIMBA v2 — Shakespeare retrain (Kaggle T4)

Retrains the **same ~30M architecture** as `dimbapeare1-30m`, now with the v2 upgrades:
**BPE** tokenizer · **v-prediction** · **AdaLN-Zero** conditioning · **latent normalization** · **self-conditioning** · normalized **logit-scaled head**.

**Before you run:**
1. `git push origin main` locally so this notebook can clone the v2 commit.
2. Kaggle: **Settings → Accelerator = GPU T4 ×1**, **Internet = ON**.

The model trains, then prints samples; the final cells report held-out perplexity + a prompted sample so you can judge *is it better*.

## 1. Install the Mamba-2 CUDA kernels + BPE deps
(Same kernels you used for the original 30M run. The build can take a few minutes.)

In [ ]:
!pip -q install causal-conv1d mamba-ssm --no-build-isolation
!pip -q install pytorch-lightning tokenizers tensorboard

## 2. Clone the v2 repo

In [ ]:
%cd /kaggle/working
!rm -rf dimba-lib-exp
!git clone --depth 1 https://github.com/devnull37/dimba-lib-exp.git
%cd dimba-lib-exp
!git log --oneline -1   # confirm you see the v2 commit

## 3. Train
Same shape as the 30M (`d_model=768`, 12 layers, `d_state=128`, `T=128`) but BPE + the v2 architecture (baked into the script defaults).

If you hit **CUDA OOM**: add `--grad-checkpoint` and/or lower `--batch-size` (e.g. 16).
If the kernel rejects `--d-state 128`: try `--d-state 64`.

In [ ]:
!python scripts/train_shakespeare.py \
  --accelerator gpu --precision 16-mixed --use-mamba-ssm \
  --d-model 768 --num-layers 12 --d-state 128 --num-steps 128 \
  --tokenizer bpe --bpe-vocab-size 4096 \
  --seq-len 256 --batch-size 32 --epochs 80 --lr 3e-4 \
  --warmup-steps 500 --patience 15 --num-workers 2 \
  --checkpoint-dir /kaggle/working/ckpt_v2

## 4. Held-out denoising perplexity
Lower at **high noise** (`t` near T) is the signal that v-prediction fixed the generative regime. Compare against the baselines it prints.

In [ ]:
!python scripts/perplexity_eval.py \
  --ckpt /kaggle/working/ckpt_v2/shakespeare1.ckpt \
  --tokenizer /kaggle/working/ckpt_v2/tokenizer.json \
  --data data/shakespeare.txt --device cuda --max-windows 400 --t-stride 8

## 5. Prompted generation (the real test)
Does it produce *words* now, not textured non-words?

In [ ]:
import sys, torch
sys.path.insert(0, '/kaggle/working/dimba-lib-exp/src')
from dimba import DIMBA
from dimba.diffusion.sampling import sample_from_model
from dimba.tokenizers.bpe import BPETokenizer

CK  = '/kaggle/working/ckpt_v2/shakespeare1.ckpt'
TOK = '/kaggle/working/ckpt_v2/tokenizer.json'
ck  = torch.load(CK, map_location='cpu', weights_only=False)
hp  = ck['hyper_parameters']
m   = DIMBA(vocab_size=hp['vocab_size'], **hp['model_config'])
m.load_state_dict({k[6:]: v for k, v in ck['state_dict'].items() if k.startswith('model.')}, strict=False)
m = m.to('cuda').eval()
tok = BPETokenizer(); tok.load(TOK)

prompt = torch.tensor([tok.encode('ROMEO:\n')], device='cuda')
for i in range(3):
    out = sample_from_model(m, prompt, seq_len=200, num_steps=128, temperature=0.8, top_p=0.95, device='cuda')
    print(f'--- sample {i+1} ---')
    print(tok.decode(out[0]))


## 6. Save the checkpoint
`/kaggle/working/ckpt_v2/` is in this notebook's **Output** — download `shakespeare1.ckpt` + `tokenizer.json`, or push the model to a HF/Kaggle dataset.

To run it on your Mac: `python scripts/perplexity_eval.py --ckpt <path> --tokenizer <path>` (loads via the pure-PyTorch `TorchMamba2` automatically).